Prompt builder

In [1]:
# Prompt Editor for JSON-based templates (drop this notebook in your prompts directory)
# Validates: template vars == declared vars (no unknowns), sane defaults, schema
# Requires: ipywidgets (bundled in Jupyter), jinja2 (optional but recommended)

from pathlib import Path
from datetime import datetime, timezone
import json, re
from ipywidgets import (
    VBox, HBox, Button, Layout, Text, Textarea, Dropdown, Output, HTML, ToggleButtons
)
from IPython.display import display

# ---------- Helpers ----------
PROMPT_DIR = Path('.').resolve()  # this notebook lives in prompts directory

def now_iso(): return datetime.now(timezone.utc).isoformat(timespec="seconds")


from jinja2 import Environment, StrictUndefined, meta
_J = Environment(undefined=StrictUndefined, trim_blocks=True, lstrip_blocks=True)

_JINJA_BUILTINS = {"loop", "cycler", "joiner", "namespace", "super", "self", "caller"}

def render_preview(template: str, values: dict) -> str:
    return _J.from_string(template or "").render(**(values or {}))


def find_template_vars(template: str) -> set[str]:
    """
    Return the *external* variables your template needs.
    - Uses Jinja2 AST to find undeclared variables
    - Normalizes 'foo.bar|filter' -> 'foo'
    - Drops Jinja builtins (loop, super, etc.)
    """
    txt = template or ""
    ast = _J.parse(txt)

    names = meta.find_undeclared_variables(ast)

    # Normalize: keep the base identifier before '.' or '|'
    cleaned = set()
    for n in names:
        base = n.split(".", 1)[0].split("|", 1)[0]
        if base and re.match(r"^[A-Za-z_]\w*$", base) and base not in _JINJA_BUILTINS:
            cleaned.add(base)
    return cleaned    


def validate_spec(spec: dict) -> tuple[bool, list[str], list[str]]:
    errs, warns = [], []

    template = spec.get("template")
    vars_    = spec.get("vars") or []
    defaults = spec.get("defaults") or {}
    tags     = spec.get("tags") or []

    # basic schema checks (omitted here for brevity)

    used = find_template_vars(template or "")
    decl = set(vars_)

    unknown = sorted(used - decl)           # used in template but not declared
    unused  = sorted(decl - used)           # declared but not used in template

    if unknown:
        errs.append("Template uses undeclared var(s): " + ", ".join(unknown))
    if unused:
        warns.append("Declared but not used: " + ", ".join(unused))

    extra_defaults = sorted(set(defaults.keys()) - decl)
    if extra_defaults:
        warns.append("Defaults provided for unknown var(s): " + ", ".join(extra_defaults))

    # try rendering with defaults (as a soft check)
    try:
        render_preview(template or "", defaults or {})
    except Exception as e:
        warns.append(f"Preview with defaults failed (ok to save): {e}")

    return (len(errs) == 0), errs, warns


def list_json_files():
    return sorted([p.name for p in PROMPT_DIR.glob("*.json")])

def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def save_json(path: Path, spec: dict) -> None:
    spec = dict(spec)
    spec.setdefault("version", 1)
    spec["updated_at"] = now_iso()
    path.write_text(json.dumps(spec, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

# ---------- Widgets ----------
w_files   = Dropdown(description="Open", options=list_json_files(), layout=Layout(width="50%"))
w_load    = Button(description="Load", layout=Layout(width="120px"))
w_newname = Text(description="New", placeholder="icd_compare.json", layout=Layout(width="50%"))
w_create  = Button(description="Create", layout=Layout(width="120px"))
w_title   = Text(description="Title", layout=Layout(width="100%"))
w_tags    = Text(description="Tags", placeholder="comma,separated", layout=Layout(width="100%"))
w_template= Textarea(description="Template", layout=Layout(width="100%", height="200px"))
w_vars    = Text(description="Vars", placeholder="comma,separated,lower_snake", layout=Layout(width="100%"))
w_defaults= Textarea(description="Defaults (JSON)", placeholder='{"scope":"LA"}', layout=Layout(width="100%", height="100px"))

w_validate= Button(description="Validate", layout=Layout(width="160px"))
w_save    = Button(description="Save JSON", layout=Layout(width="160px"), disabled=True)
w_preview = Textarea(description="Preview", layout=Layout(width="100%", height="160px"))
w_msgs    = Output(layout=Layout(border="1px solid #eee", padding="6px"))

# state
state = {"path": None, "spec": {}}

# ---------- Actions ----------
def sync_widgets_from_spec(spec: dict):
    w_title.value    = spec.get("title","")
    w_tags.value     = ",".join(spec.get("tags") or [])
    w_template.value = spec.get("template","")
    w_vars.value     = ",".join(spec.get("vars") or [])
    defaults = spec.get("defaults") or {}
    try:
        w_defaults.value = json.dumps(defaults, ensure_ascii=False, indent=2)
    except Exception:
        w_defaults.value = "{}"

def collect_spec_from_widgets() -> dict:
    # parse vars + defaults
    vars_list = [v.strip() for v in w_vars.value.split(",") if v.strip()]
    try:
        defaults = json.loads(w_defaults.value.strip() or "{}")
    except Exception:
        defaults = {}
    return {
        "version": 1,
        "title": w_title.value.strip(),
        "template": w_template.value,
        "vars": vars_list,
        "defaults": defaults,
        "tags": [t.strip() for t in w_tags.value.split(",") if t.strip()],
    }

def do_validate(_=None, for_save=False):
    w_msgs.clear_output()
    spec = collect_spec_from_widgets()
    ok, errs, warns = validate_spec(spec)
    w_save.disabled = not ok

    # ----- PREVIEW (fill missing vars with placeholders so it never errors) -----
    template = spec["template"]
    defaults = spec.get("defaults") or {}
    declared = list(spec.get("vars") or [])
    preview_values = dict(defaults)
    for k in declared:
        if k not in preview_values:
            preview_values[k] = f"{{{{{k}}}}}"   # placeholder for missing

    try:
        w_preview.value = render_preview(template, preview_values)
    except Exception as e:
        # Only hit if template has a syntax error, not missing vars
        w_preview.value = f"(preview failed) {e}"
        # Keep this as a warning so Save still works if schema is otherwise valid
        warns.append(f"Preview failed: {e}")

    # messages
    with w_msgs:
        if ok:
            print("✅ Valid.")
        if errs:
            print("❌ Errors:")
            for e in errs: print("  -", e)
        if warns:
            print("⚠️ Warnings:")
            for w in warns: print("  -", w)
    return ok

def do_save(_=None):
    if not do_validate(for_save=True):
        return
    name = (w_newname.value or "").strip()
    if not name and not state["path"]:
        with w_msgs:
            w_msgs.clear_output(); print("❌ Provide a New file name or load an existing file.")
        return
    path = state["path"] or PROMPT_DIR / (name if name.endswith(".json") else f"{name}.json")
    spec = collect_spec_from_widgets()
    save_json(path, spec)
    state["path"] = path
    w_files.options = list_json_files()
    with w_msgs:
        print(f"💾 Saved {path.name} at {now_iso()}")

def do_load(_=None):
    sel = w_files.value
    if not sel:
        with w_msgs:
            w_msgs.clear_output(); print("⚠️ No file selected.")
        return
    path = PROMPT_DIR / sel
    try:
        spec = load_json(path)
    except Exception as e:
        with w_msgs:
            w_msgs.clear_output(); print(f"❌ Load failed: {e}")
        return
    state["path"] = path
    sync_widgets_from_spec(spec)
    w_newname.value = ""  # avoid accidental overwrite rename
    with w_msgs:
        w_msgs.clear_output(); print(f"📂 Loaded {path.name}")

def do_create(_=None):
    # start a minimal valid spec
    spec = {
        "version": 1,
        "title": "Conponent Report",
        "template": "Generate a report {{component_a}} vs {{component_b}} at {{scope}} level.",
        "vars": ["component_a","component_b","scope"],
        "defaults": {"scope":"LA"},
        "tags": []
    }
    state["path"] = None
    sync_widgets_from_spec(spec)
    with w_msgs:
        w_msgs.clear_output(); print("📝 Draft created (not saved yet).")

# wire buttons
w_validate.on_click(do_validate)
w_save.on_click(do_save)
w_load.on_click(do_load)
w_create.on_click(do_create)

# ---------- Layout ----------
display(
    VBox([
        HBox([w_files, w_load, w_newname, w_create], layout=Layout(align_items="flex-end")),
        w_title, w_tags, w_template, w_vars, w_defaults,
        HBox([w_validate, w_save], layout=Layout(align_items="flex-end")),
        w_preview, w_msgs
    ], layout=Layout(width="100%"))
)

# Initial scan
w_files.options = list_json_files()
